### Practice: Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [1]:
# for kaggle
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
%pip install --upgrade peft bitsandbytes

import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 886.7/886.7 kB 8.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 72.8 MB/s  0:00:00 eta 0:00:01m
  Attempting uninstall: peft━━━━━━━━━━━━━━━━━━━━ 0/2 [bitsandbytes]
    Found existing installation: peft 0.21.2 0/2 [bitsandbytes]
    Uninstalling peft-0.21.2:━━━━━━━━━━━━━━━ 0/2 [bitsandbytes]
      Successfully uninstalled peft-0.21.2━━━━━━━━━━━━━━━━━━━ 1/2 [peft]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [peft]1/2 [peft]
Note: you may need to restart the kernel to use updated packages.


In [3]:
model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id

# the main model weights are loaded in 4-bit precision - but we can still tune LoRAs later
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float32,
    quantization_config=transformers.BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4"),
)
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
# more on gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

config.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.43k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.07GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

### Prompt tuning: the story of a fox (1 point)

![img](https://i.imgur.com/Ux3qQAu.png) (source: theodd1souts.fandom.com)

In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: A quick brown fox jumps over the lazy dog. The quick brown fox


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.5625, device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<NllLossBackward0>)


Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [ ]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace model's original word embeddings with a layer - THIS layer
     - that inserts trainable prompts instead of the first N token embeddings. """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True)

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), "don't forget to prepend several BOS tokens to input_ids"

        # Your task: embed input_ids, but replace the first :num_prompts: tokens with self.learnable_prompts
        # This is because we will prepend :num_prompts: padding tokens at the beginning

        # After you are done, you must produce a word embedding vector for each token in input_ids,
        # except that the first :num_prompts: vectors should equal learnable_prompts;
        # any additional vectors after first :num_prompts: ones should be embedded as usual
        # Note: since you're dealing with trainable params, please torch.cat instead of item assignment
        batch_size, seq_len = input_ids.shape
        embs = self.original_word_embeddings(input_ids) # (batch, seq_len, emb_dim)
        prompt_embs = torch.cat(
            [
                self.learnable_prompts.expand(batch_size, -1, -1),
                embs[:, self.num_prompts:, :]
            ],
            dim=1
        )
        return prompt_embs

In [ ]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


/tmp/ipykernel_9620/1152930240.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [ ]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

with torch.amp.autocast('cuda', dtype=torch.bfloat16):
    outputs = model(**batch)
next_word_logits = outputs.logits[:, num_prompts : -1, :]
true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
print("Loss:", loss)


for i in range(50):
    loss.backward()
    opt.step()
    opt.zero_grad()

    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        outputs = model(**batch)
    next_word_logits = outputs.logits[:, num_prompts : -1, :]
    loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
    if i % 10 == 0:
        print(loss.item())


assert loss.item() <= 0.1
print("Good job!")

Loss: tensor(2.5625, device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<NllLossBackward0>)
2.484375
1.5859375
0.10498046875
0.006317138671875
0.002655029296875
Good job!


In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# if you did everything right, the model will deny that the fox jumped over the lazy dog


Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway


### Using HuggingFace PEFT

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [3]:
import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 4717917184


In [ ]:
# Your task: optimize the PEFT-wrapped model to achieve next token prediction loss < 0.1, but this time using PEFT
# Please note: you no longer need to prepend PAD tokens, but you still need to skip :num_virtual_tokens: first logits.
# Finally, generate the sentence to make sure that the model learned the truth.

In [ ]:
# Feel free to structure your code as you see fit - as long as it's legible :)

In [4]:
def get_outputs(model, inputs, max_new_tokens=20):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    )
    return outputs

In [5]:
prompt = 'A quick brown fox'
inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = get_outputs(model, inputs)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['A quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog. The quick brown fox']


In [6]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)

In [7]:
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

class MyVeryLargeDataset(Dataset):
    def __init__(self, the_truth: str):
        self.the_truth = the_truth

    def __getitem__(self, idx):
        if idx > 0:
            raise IndexError
        return dict(tokenizer(self.the_truth))

    def __len__(self):
        return 1

In [8]:
dataset = MyVeryLargeDataset(the_truth)

In [9]:
training_args = TrainingArguments(
    num_train_epochs=75,
    learning_rate=1e-2,
    save_strategy='no',
    logging_strategy='steps',
    logging_steps=15
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

In [10]:
trainer.train()

Step,Training Loss
15,3.171217
30,1.591470


Step,Training Loss
15,3.171217
30,1.591470
45,0.198154
60,0.004442
75,0.001940


TrainOutput(global_step=75, training_loss=0.9934446051716804, metrics={'train_runtime': 170.8112, 'train_samples_per_second': 0.439, 'train_steps_per_second': 0.439, 'total_flos': 68115649536000.0, 'train_loss': 0.9934446051716804, 'epoch': 75.0})

In [11]:
trainer.evaluate(eval_dataset=dataset)["eval_loss"]

Training Loss,Validation Loss,Step
0.001940,0.001630,75


0.001630135579034686

In [12]:
prompt = 'A quick brown fox'
inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = get_outputs(model, inputs)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!']


### Parameter-efficient finetuning with LoRA (1 point)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [2]:
import peft  # v-- re-load the notebook to remove any previous PEFT tuners, run cell 1 and this
model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float32,
    quantization_config=transformers.BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4"),
).to(device)
model = peft.prepare_model_for_kbit_training(model, use_gradient_checkpointing=True) # < checkpoints and input grads

config.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.43k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.07GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

In [3]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Apply self.module and LoRA adapter, return the sum (self.module outputs + adapter outputs)
        return self.module(input) + (input @ self.adapter_A) @ self.adapter_B

In [4]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in attention blocks. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

__Note:__ please scroll down for the homework task

In [5]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'DecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) > 0, "Did not add any LoRA layers!"

In [6]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False).to(device)
# test a single training step, make sure we get meaningful gradients
with torch.cuda.amp.autocast(dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

/tmp/ipykernel_475/2281301266.py:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float32):


Grad check successful, well done!


### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [24]:
# checking if the model can learn. Change max_steps for proper training
import datasets
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # silence a warning from HF trainer

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=1,
        # note: if you want larger batch size, increase gradient_accumulation_steps
        warmup_steps=250, max_steps=10, learning_rate=2e-4, fp16=True,
        logging_steps=10, output_dir='outputs', save_strategy='no'),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl: reconstructing file:   0%|          |  0.00B /  647kB            

quotes.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Step,Training Loss
10,1.175048


TrainOutput(global_step=10, training_loss=1.1750478744506836, metrics={'train_runtime': 14.0104, 'train_samples_per_second': 1.428, 'train_steps_per_second': 0.714, 'total_flos': 36308127485952.0, 'train_loss': 1.1750478744506836, 'epoch': 0.625})

### Final task: *actually* train the model (3 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter validation subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

In [5]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # feel free to add a few more that are not 100% assiciated with Python

# <A WHOLE LOT OF YOUR CODE>
# generate baseline samples with the selected prompts before finetuning
# please feel free to use transformers.Trainer (as above) or your custom training code
# after the training concludes, please show examples of text generated by your model. It is expected to look like Python code fragments
# print the generation examples nicely (suggestion: use pandas or HTML) for easier comparison
# note: your LoRA-enhanced model can run generation the same way as the non-trained model (above)

In [6]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import peft

In [7]:
model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name, padding_side='left')
tokenizer.pad_token_id = tokenizer.eos_token_id
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map=0, low_cpu_mem_usage=True, dtype=torch.float32,
    quantization_config=transformers.BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4"),
)
model = peft.prepare_model_for_kbit_training(model, use_gradient_checkpointing=True) # < checkpoints and input grads

config.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.43k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.07GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

In [8]:
import datasets


train_dataset = datasets.load_dataset("codeparrot/codeparrot-clean-valid", split="train[:10%]")
eval_dataset = datasets.load_dataset("codeparrot/codeparrot-clean-valid", split="train[-50:]")

README.md:   0%|          | 0.00/401 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


file-000000000054.json.gz: reconstructing file:   0%|          |  0.00B /  142MB            

file-000000000054.json.gz: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/61373 [00:00<?, ? examples/s]

Repo card metadata block was not found. Setting CardData to empty.


In [9]:
eval_dataset

Dataset({
    features: ['repo_name', 'path', 'copies', 'size', 'content', 'license', 'hash', 'line_mean', 'line_max', 'alpha_frac', 'autogenerated'],
    num_rows: 50
})

In [10]:
def tokenize(sample):
    return tokenizer(sample['content'], truncation=True, max_length=512)

In [11]:
data_train = train_dataset.map(tokenize, batched=True)
data_eval = eval_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/6137 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [12]:
def get_outputs(model, inputs, max_new_tokens=20):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    )
    return outputs

In [13]:
inputs = tokenizer(prompts, return_tensors='pt', padding=True)

In [14]:
base_outputs = get_outputs(model, inputs, max_new_tokens=128)

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [15]:
base_outputs_decoded = tokenizer.batch_decode(base_outputs, skip_special_tokens=True)
print(base_outputs_decoded)

['from \\eqref{eq:decomp} and \\eqref{eq:decomp2} that\n\\begin{equation}\\label{eq:decomp3}\n\\begin{aligned}\n&\\int_{\\mathbb{R}^d}\\left(\\frac{\\partial^2}{\\partial x_1^2}+\\cdots+\\frac{\\partial^2}{\\partial x_d^2}\\right)\\left(\\frac{\\partial^2}{\\partial x_1^2}+\\cdots+\\frac{\\partial^2}{\\partial x_d^2}\\right)\\left(\\frac', "import { async, ComponentFixture, TestBed } from '@angular/core/testing';\n\nimport { AdminDashboardComponent } from './admin-dashboard.component';\nimport { RouterTestingModule } from '@angular/router/testing';\nimport { HttpClientTestingModule } from '@angular/common/http/testing';\nimport { FormsModule } from '@angular/forms';\nimport { AdminService } from '../admin.service';\nimport { AdminServiceMock } from '../admin.service.mock';\nimport { AdminDashboardService } from './admin-dashboard.service';\nimport { AdminDashboardServiceMock } from './admin-dashboard.service.mock';\nimport { AdminDashboardComponentMock } from './admin-dashboard.compone

In [16]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear4bit(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNor

In [17]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'DecoderLayer' in repr(type(module)):
        module.mlp.up_proj = LoRALayer(module.mlp.up_proj, rank=lora_rank).to(device)
        module.mlp.down_proj = LoRALayer(module.mlp.down_proj, rank=lora_rank).to(device)
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) > 0, "Did not add any LoRA layers!"

In [18]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): LoRALayer(
            (module): Linear4bit(in_features=4096, out_features=4096, bias=False)
          )
          (k_proj): LoRALayer(
            (module): Linear4bit(in_features=4096, out_features=1024, bias=False)
          )
          (v_proj): LoRALayer(
            (module): Linear4bit(in_features=4096, out_features=1024, bias=False)
          )
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (up_proj): LoRALayer(
            (module): Linear4bit(in_features=4096, out_features=12288, bias=False)
 

In [21]:
model._hf_peft_config_loaded = True  # silence a warning from HF trainer


training_args = TrainingArguments(
    max_steps=100,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=0.05,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    fp16=False,
    logging_strategy='steps',
    logging_steps=5,
    eval_on_start=True,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy="no"
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data_train,
    eval_dataset=data_eval,
    data_collator=data_collator
)

In [22]:
trainer.train()

Step,Training Loss,Validation Loss
0,No log,1.020596
50,1.075050,1.006348
100,0.946657,1.001132


TrainOutput(global_step=100, training_loss=0.9944313097000123, metrics={'train_runtime': 5289.9522, 'train_samples_per_second': 0.076, 'train_steps_per_second': 0.019, 'total_flos': 8627730168576000.0, 'train_loss': 0.9944313097000123, 'epoch': 0.06517842594101353})

In [23]:
peft_outputs = get_outputs(model, inputs, max_new_tokens=64)
peft_outputs_decoded = tokenizer.batch_decode(peft_outputs, skip_special_tokens=True)
print(peft_outputs_decoded)

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['The following is a single-choice question from a Chinese engineering exam. Please select the correct answer.\nThe main function of the 104-type distribution valve is to control the charging and discharging of the brake cylinder based on the pressure changes in the train pipe, thereby achieving the braking and relief of the vehicle. Is', "import {\n  Component,\n  OnInit,\n  Input,\n  Output,\n  EventEmitter,\n  OnChanges,\n  SimpleChanges,\n  ViewChild,\n  ElementRef,\n  AfterViewInit,\n  OnDestroy,\n} from '@angular/core';\nimport { FormGroup, FormControl, Validators } from '@angular/forms';\nimport { Router } from '@angular", 'from django.db import models\nfrom django.contrib.auth.models import User\nfrom django.utils import timezone\nfrom django.utils.text import slugify\nfrom django.urls import reverse\nfrom django.core.validators import MaxValueValidator, MinValueValidator\nfrom django.core.exceptions import ValidationError\nfrom django.db.models.signals import pre_save\nfrom dj

In [26]:
# This template helps to compare generated code samples in pretty table form
# feel free to present your work in other forms

from IPython.display import HTML, display
table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">PROMPT</th>
    <th style="text-align: center; border:1px solid black">BEFORE</th>
    <th style="text-align: center; border:1px solid black">AFTER</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>'''

rows = []

for prompt, base_output, peft_output in zip(prompts, base_outputs_decoded, peft_outputs_decoded):
    # replace placeholders in the format() arguments
    rows.append(row_template.format(prompt, base_output, peft_output))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
``,from \eqref{eq:decomp} and \eqref{eq:decomp2} that \begin{equation}\label{eq:decomp3} \begin{aligned} &\int_{\mathbb{R}^d}\left(\frac{\partial^2}{\partial x_1^2}+\cdots+\frac{\partial^2}{\partial x_d^2}\right)\left(\frac{\partial^2}{\partial x_1^2}+\cdots+\frac{\partial^2}{\partial x_d^2}\right)\left(\frac,"The following is a single-choice question from a Chinese engineering exam. Please select the correct answer. The main function of the 104-type distribution valve is to control the charging and discharging of the brake cylinder based on the pressure changes in the train pipe, thereby achieving the braking and relief of the vehicle. Is"
`import`,"import { async, ComponentFixture, TestBed } from '@angular/core/testing'; import { AdminDashboardComponent } from './admin-dashboard.component'; import { RouterTestingModule } from '@angular/router/testing'; import { HttpClientTestingModule } from '@angular/common/http/testing'; import { FormsModule } from '@angular/forms'; import { AdminService } from '../admin.service'; import { AdminServiceMock } from '../admin.service.mock'; import { AdminDashboardService } from './admin-dashboard.service'; import { AdminDashboardServiceMock } from './admin-dashboard.service.mock'; import { AdminDashboardComponentMock } from './admin-dashboard.component.mock'; import { AdminDashboardModule } from './admin","import { Component, OnInit, Input, Output, EventEmitter, OnChanges, SimpleChanges, ViewChild, ElementRef, AfterViewInit, OnDestroy, } from '@angular/core'; import { FormGroup, FormControl, Validators } from '@angular/forms'; import { Router } from '@angular"
`from`,from \eqref{eq:decomp} and \eqref{eq:decomp2} that \begin{equation}\label{eq:decomp3} \begin{aligned} &\int_{\mathbb{R}^d}\left(\frac{\partial^2}{\partial x_1^2}+\cdots+\frac{\partial^2}{\partial x_d^2}\right)\left(\frac{\partial^2}{\partial x_1^2}+\cdots+\frac{\partial^2}{\partial x_d^2}\right)\left(\frac{\,"from django.db import models from django.contrib.auth.models import User from django.utils import timezone from django.utils.text import slugify from django.urls import reverse from django.core.validators import MaxValueValidator, MinValueValidator from django.core.exceptions import ValidationError from django.db.models.signals import pre_save from django"
`while`,while \n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t,while *p++ = *q++; // 1 while (*p++ = *q++); // 2 while (*p++ = *q++); // 3 while (*p++ = *q++); // 4 while (*p++ = *q++); // 5
`try`,try \n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t,"try: from setuptools import setup except ImportError: from distutils.core import setup setup( name='django-redis-cache', version='0.1.1', description='Django cache backend using Redis', author='Dmitry Shulga', author_email='dmitry"
`if`,"if \(\frac{a}{b} = \frac{c}{d}\), then \(a \cdot d = b \cdot c\). This is because if the ratios are equal, the cross products must also be equal. For example, if \(\frac{2}{3} = \frac{4}{6}\), then \(2 \cdot 6 = 3 \cdot 4\), which simplifies to \(12 = 12\). To apply this property to solve problems, consider the following example: Suppose you have the equation \(\frac{x}{4} = \frac{3}{","if __name__ == '__main__': n = int(input()) for i in range(1, n+1): print(i, end="""")"
`for`,"for \(\alpha \in (0,1)\), where \(a,b,c,d\in \mathbb{R}\) and \(a,b,c,d\neq 0\). The authors give a simple proof of this inequality and present some applications.","for _ in range(int(input())): n = int(input()) a = list(map(int, input().split())) b = list(map(int, input().split())) c = list(ma

If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.